# Stage 3 — Pairs Trading: Intraday Backtest [v1.0-FRESH]
**Method**: Kalman Filter Z-Score Signal with fixed Q/R from Stage 2 EM
**Universe**: 41 pairs (5 ≤ HL ≤ 120 min from Stage 2)
**Signal**: Rolling 3750-bar Z-score on session-continuous innovations
**Output**: pairs_stage3_backtest.csv ranked by Calmar Ratio


## Cell 0 — Version Check + Path Discovery


In [ ]:
NB_VERSION = "v1.0-FRESH"
print(f"Notebook version: {NB_VERSION}")

import os, glob
print("\n=== /kaggle/input ===")
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        fp = os.path.join(root, f)
        print(f"  {fp}  ({os.path.getsize(fp)/1e6:.1f} MB)")

hits_db   = glob.glob('/kaggle/input/**/*.sqlite',                    recursive=True)
hits_s2   = glob.glob('/kaggle/input/**/pairs_stage2_kalman_ou.csv',  recursive=True)
hits_s1   = glob.glob('/kaggle/input/**/pairs_top500.csv',            recursive=True)
if not hits_db:  raise FileNotFoundError("No .sqlite found")
if not hits_s2:  raise FileNotFoundError("No pairs_stage2_kalman_ou.csv found")
if not hits_s1:  raise FileNotFoundError("No pairs_top500.csv found")
DB_PATH   = hits_db[0]
S2_CSV    = hits_s2[0]
S1_CSV    = hits_s1[0]
print(f"\nDB_PATH = {DB_PATH}")
print(f"S2_CSV  = {S2_CSV}")
print(f"S1_CSV  = {S1_CSV}")


## Cell 1 — Imports and Constants


In [ ]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"]      = "1"

import sqlite3, datetime, warnings, traceback, time
from collections import deque
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

# ── Constants ──
MARKET_OPEN    = datetime.time(9, 15)
MARKET_CLOSE   = datetime.time(15, 29)   # last valid bar
FORCE_EXIT_TIME = datetime.time(15, 28)  # force-exit at this bar
BARS_PER_DAY   = 375
WARMUP_BARS    = 3_750                   # 10 trading days
MIN_TOTAL_BARS = WARMUP_BARS + BARS_PER_DAY  # must have at least 1 live day
CAPITAL        = 10_000.0               # Rs per pair (fixed, not compounding)
Z_ENTRY        = 2.0                    # |z| >= 2 triggers entry
Z_EXIT         = 0.0                    # z crosses 0 triggers reversion exit
HL_MIN         = 5.0                    # Stage 3 filter: min half-life minutes
HL_MAX         = 120.0                  # Stage 3 filter: max half-life minutes

print(f"WARMUP_BARS : {WARMUP_BARS} ({WARMUP_BARS//BARS_PER_DAY} trading days)")
print(f"CAPITAL     : Rs {CAPITAL:,.0f} per pair")
print(f"Z_ENTRY     : {Z_ENTRY}")
print(f"Z_EXIT      : {Z_EXIT}")
print(f"HL filter   : {HL_MIN} <= HL <= {HL_MAX} minutes")


## Cell 2 — Load Stage 1 + Stage 2, Apply Half-Life Filter


In [ ]:
s1_df = pd.read_csv(S1_CSV)
s2_df = pd.read_csv(S2_CSV)

# Stage 3 filter: HL only (no ADF filter per design spec)
s3_df = s2_df[
    (s2_df["half_life_minutes"] >= HL_MIN) &
    (s2_df["half_life_minutes"] <= HL_MAX)
].copy().reset_index(drop=True)

# pearson_rho and stage1_rank already exist in Stage 2 CSV — no merge needed.
# Merge any extra Stage 1 columns gracefully if they exist.
s1_extra = [c for c in s1_df.columns
            if c not in ["symbol_a","symbol_b"] and c not in s3_df.columns]
if s1_extra:
    s3_df = s3_df.merge(s1_df[["symbol_a","symbol_b"]+s1_extra],
                        on=["symbol_a","symbol_b"], how="left")

print(f"Stage 2 total rows    : {len(s2_df)}")
print(f"Stage 3 filtered pairs: {len(s3_df)}  (HL: {HL_MIN}-{HL_MAX} min)")
print(s3_df[["symbol_a","symbol_b","half_life_minutes","pearson_rho","stage1_rank","Q_beta","Q_alpha","R"]].to_string())


## Cell 3 — Price Loader: Session-Continuous Series
Loads 1-min OHLCV from SQLite. Drops ALL non-trading hours and non-trading days.
Stitches remaining bars as a clean continuous array — no overnight gaps.


In [ ]:
def load_session_continuous(db_path, symbol_a, symbol_b):
    """
    Returns: (ln_a, ln_b, timestamps) as np.ndarrays
    All arrays are session-continuous: only 09:15-15:29 IST bars,
    weekends and NSE holidays implicitly dropped (they have no data),
    stitched directly — no gap-fill, no overnight data ever present.
    """
    conn = sqlite3.connect(db_path)
    query = """
        SELECT timestamp, symbol, close
        FROM ohlcv_1min
        WHERE symbol IN (?, ?)
          AND close IS NOT NULL
          AND close > 0
        ORDER BY timestamp
    """
    df = pd.read_sql_query(query, conn, params=(symbol_a, symbol_b))
    conn.close()

    # FIXED: unit="s" for correct parsing, tz_convert for IST
    df["timestamp"] = pd.to_datetime(df["timestamp"], unit="s", utc=True).dt.tz_convert("Asia/Kolkata")

    # ── MANDATORY: drop all non-trading hours ──
    t = df["timestamp"].dt.time
    mask = (t >= datetime.time(9, 15)) & (t <= datetime.time(15, 29))
    df = df[mask].copy()
    # Non-trading days (weekends, holidays) have no rows in the DB —
    # dropping non-trading hours is sufficient; gaps are already absent.

    # Pivot: one row per timestamp, columns = symbols
    pivot = df.pivot_table(index="timestamp", columns="symbol", values="close", aggfunc="last")

    # Assert both symbols present
    if symbol_a not in pivot.columns or symbol_b not in pivot.columns:
        raise ValueError(f"Missing column for {symbol_a} or {symbol_b}")

    # Inner-join: only timestamps where BOTH symbols have data
    pivot = pivot[[symbol_a, symbol_b]].dropna(how="any")

    # Sort chronologically
    pivot = pivot.sort_index()

    # Sanity: assert all timestamps are within market hours
    times = pivot.index.time
    assert all((t >= datetime.time(9,15)) and (t <= datetime.time(15,29)) for t in times), \
        "Non-market-hours timestamp found after filter!"

    ln_a = np.log(pivot[symbol_a].values).astype(np.float64)
    ln_b = np.log(pivot[symbol_b].values).astype(np.float64)
    timestamps = pivot.index

    return ln_a, ln_b, timestamps


# Test with first pair
row0 = s3_df.iloc[0]
la, lb, ts = load_session_continuous(DB_PATH, row0.symbol_a, row0.symbol_b)
print(f"Test pair: {row0.symbol_a} / {row0.symbol_b}")
print(f"Total bars in session-continuous series: {len(la)}")
print(f"First bar: {ts[0]}  |  Last bar: {ts[-1]}")
print(f"Min time: {min(t.time() for t in ts)}  |  Max time: {max(t.time() for t in ts)}")


## Cell 4 — Online Kalman Filter (Fixed Q, R from Stage 2 EM)
Q and R are NOT re-estimated. They are loaded directly from Stage 2 EM output.
This prevents look-ahead bias. The filter runs as a pure online predictor.


In [ ]:
def kalman_filter_fixed(ln_a, ln_b, beta0, alpha0, P0, Q_beta, Q_alpha, R):
    """
    Online Kalman filter with FIXED Q and R (from Stage 2 EM).
    State: theta = [beta, alpha]
    Obs:   y_t = beta*ln_b_t + alpha + v_t,  v_t ~ N(0, R)
    Trans: theta_t = theta_{t-1} + w_t,      w_t ~ N(0, Q)

    Returns:
        innovations: np.ndarray of shape (T,) — raw spread e_t at each bar
        betas      : np.ndarray of shape (T,) — filtered beta
        alphas     : np.ndarray of shape (T,) — filtered alpha
    """
    T = len(ln_a)
    Q = np.array([[Q_beta, 0.0], [0.0, Q_alpha]], dtype=np.float64)

    theta = np.array([beta0, alpha0], dtype=np.float64)
    P     = P0.copy()

    innovations = np.empty(T, dtype=np.float64)
    betas       = np.empty(T, dtype=np.float64)
    alphas      = np.empty(T, dtype=np.float64)

    for t in range(T):
        H = np.array([ln_b[t], 1.0], dtype=np.float64)  # observation vector

        # Predict
        theta_pred = theta.copy()
        P_pred     = P + Q

        # Innovation
        y_t   = ln_a[t]
        e_t   = y_t - H @ theta_pred
        S_t   = H @ P_pred @ H + R
        K_t   = (P_pred @ H) / S_t

        # Update
        theta = theta_pred + K_t * e_t
        P     = (np.eye(2) - np.outer(K_t, H)) @ P_pred

        innovations[t] = e_t
        betas[t]       = theta[0]
        alphas[t]      = theta[1]

    return innovations, betas, alphas


def ols_init(ln_a, ln_b):
    """OLS estimate of beta and alpha over provided bars."""
    X = np.column_stack([ln_b, np.ones(len(ln_b))])
    result = np.linalg.lstsq(X, ln_a, rcond=None)
    coeffs = result[0]
    beta0, alpha0 = coeffs[0], coeffs[1]
    # Covariance of OLS estimate (inflated 10x for Kalman init)
    resid = ln_a - X @ coeffs
    sigma2 = np.var(resid)
    XtX_inv = np.linalg.inv(X.T @ X)
    P0 = sigma2 * XtX_inv * 10.0
    return beta0, alpha0, P0

print("Kalman filter + OLS init functions defined.")


## Cell 5 — Leader/Lagger Detection
Uses 1-bar lagged cross-correlation on log-returns over the warm-up window.
The lagging asset is fixed as the sole traded asset for the entire backtest.


In [ ]:
def detect_lagger(ln_a, ln_b, warmup_bars):
    """
    Returns: (lagger, leader) where each is "a" or "b"
    Method: 1-bar lagged cross-correlation on log-returns over warm-up window.
      corr_a_leads = corr(ret_a[1:], ret_b[:-1])  -- does a lead b?
      corr_b_leads = corr(ret_b[1:], ret_a[:-1])  -- does b lead a?
    Higher absolute correlation identifies the leader.
    The NON-leader = lagger = traded asset.
    """
    ret_a = np.diff(ln_a[:warmup_bars])
    ret_b = np.diff(ln_b[:warmup_bars])

    # corr of a_t with b_{t-1}: if high, a responds to b => b leads, a lags
    corr_a_lags = np.corrcoef(ret_a[1:], ret_b[:-1])[0, 1]
    # corr of b_t with a_{t-1}: if high, b responds to a => a leads, b lags
    corr_b_lags = np.corrcoef(ret_b[1:], ret_a[:-1])[0, 1]

    if abs(corr_b_lags) >= abs(corr_a_lags):
        # b responds more to lagged a => b is the lagger
        return "b", "a"
    else:
        # a responds more to lagged b => a is the lagger
        return "a", "b"

print("Leader/lagger detection function defined.")


## Cell 6 — Zerodha MIS Intraday Equity Fee Calculator
Full breakdown: Brokerage (₹20/order), STT, Exchange charges, GST, SEBI, Stamp Duty.
All rates as per Zerodha for NSE Equity Intraday (MIS) segment.


In [ ]:
def calc_zerodha_mis_fees(qty, entry_price, exit_price, is_long):
    """
    Computes total Zerodha MIS fees for one round-trip trade.

    Zerodha MIS Equity rates:
      Brokerage        : Rs 20 flat per executed order (2 orders = Rs 40 total)
      STT              : 0.025% on sell-side turnover only
      Exchange charge  : 0.00345% on total turnover (NSE equity segment)
      GST              : 18% on (brokerage + exchange_charge)
      SEBI charges     : Rs 10 per crore = 10/1e7 per rupee of turnover
      Stamp duty       : 0.003% on buy-side turnover only

    is_long=True  => entry is BUY, exit is SELL
    is_long=False => entry is SELL (short), exit is BUY (cover)
    """
    entry_turnover = qty * entry_price
    exit_turnover  = qty * exit_price
    total_turnover = entry_turnover + exit_turnover

    # Brokerage: Rs 20 per order leg, 2 legs per round trip
    brokerage = 40.0

    # STT: 0.025% on SELL-side turnover only
    if is_long:
        stt_turnover = exit_turnover   # sell side is the exit for longs
    else:
        stt_turnover = entry_turnover  # sell side is the entry for shorts
    stt = 0.00025 * stt_turnover

    # Exchange transaction charge: 0.00345% of total turnover
    exchange_charge = 0.0000345 * total_turnover

    # GST: 18% on (brokerage + exchange_charge)
    gst = 0.18 * (brokerage + exchange_charge)

    # SEBI charges: Rs 10 per crore of turnover
    sebi = (10.0 / 1e7) * total_turnover

    # Stamp duty: 0.003% of BUY-side turnover only
    if is_long:
        stamp_turnover = entry_turnover   # buy side is the entry for longs
    else:
        stamp_turnover = exit_turnover    # buy side is the exit (cover) for shorts
    stamp = 0.00003 * stamp_turnover

    total_fees = brokerage + stt + exchange_charge + gst + sebi + stamp

    return {
        "brokerage"      : brokerage,
        "stt"            : stt,
        "exchange_charge": exchange_charge,
        "gst"            : gst,
        "sebi"           : sebi,
        "stamp"          : stamp,
        "total_fees"     : total_fees,
    }


# Sanity check at Rs 10,000 turnover
test_fees = calc_zerodha_mis_fees(qty=100, entry_price=50.0, exit_price=50.0, is_long=True)
print("Fee breakdown for Rs 10,000 turnover (round trip):")
for k, v in test_fees.items():
    print(f"  {k:<20}: Rs {v:.4f}")


## Cell 7 — Single-Pair Backtest Engine
Full walk-forward backtest for one pair. Warm-up → Kalman → Rolling Z → Signals → Fees.


In [ ]:
def backtest_pair(row, db_path):
    """
    Runs the full Stage 3 backtest for a single pair.
    Returns a dict of performance metrics.
    """
    t_start = time.time()
    sym_a, sym_b = row["symbol_a"], row["symbol_b"]
    Q_beta  = float(row["Q_beta"])
    Q_alpha = float(row["Q_alpha"])
    R       = float(row["R"])
    hl_mins = float(row["half_life_minutes"])
    hl_bars = int(np.ceil(hl_mins))  # half-life timeout in bars

    base = dict(
        symbol_a=sym_a, symbol_b=sym_b,
        Q_beta=Q_beta, Q_alpha=Q_alpha, R=R,
        half_life_minutes=hl_mins,
        capital_inr=CAPITAL,
        skipped=False, skip_reason="", error=""
    )

    try:
        # ── Load session-continuous series ──
        ln_a, ln_b, timestamps = load_session_continuous(db_path, sym_a, sym_b)
        T = len(ln_a)

        if T < MIN_TOTAL_BARS:
            base.update(skipped=True, skip_reason=f"Only {T} bars < {MIN_TOTAL_BARS} required")
            return base

        # ── Warm-up: OLS init ──
        beta0, alpha0, P0 = ols_init(ln_a[:WARMUP_BARS], ln_b[:WARMUP_BARS])

        # ── Leader/Lagger detection (fixed once from warm-up) ──
        lagger_side, leader_side = detect_lagger(ln_a, ln_b, WARMUP_BARS)
        lagging_asset = sym_a if lagger_side == "a" else sym_b
        leader_asset  = sym_b if lagger_side == "a" else sym_a

        # ── Full Kalman filter over entire series (warm-up + live) ──
        innovations, betas, alphas = kalman_filter_fixed(
            ln_a, ln_b, beta0, alpha0, P0, Q_beta, Q_alpha, R
        )

        # ── Rolling Z-score via deque (session-continuous bar count only) ──
        win = deque(maxlen=WARMUP_BARS)
        # Prime the deque with warm-up innovations
        for i in range(WARMUP_BARS):
            win.append(innovations[i])

        z_scores = np.full(T, np.nan)
        for i in range(WARMUP_BARS, T):
            win.append(innovations[i])
            mu_w  = sum(win) / len(win)
            # Welford variance using sum of squares
            var_w = sum((x - mu_w)**2 for x in win) / max(len(win) - 1, 1)
            sigma_w = var_w**0.5
            if sigma_w > 1e-10:
                z_scores[i] = (innovations[i] - mu_w) / sigma_w

        # ── Backtest: walk forward through live bars ──
        trades = []
        in_trade    = False
        entry_bar   = None
        entry_price = None
        entry_z     = None
        is_long     = None  # True = bought lagging asset, False = shorted
        qty         = 0

        # Price array for lagging asset (raw price recovered from log_price)
        if lagger_side == "a":
            lag_prices = np.exp(ln_a)
        else:
            lag_prices = np.exp(ln_b)

        for i in range(WARMUP_BARS, T):
            z = z_scores[i]
            if np.isnan(z):
                continue

            bar_time = timestamps[i].time()
            price    = lag_prices[i]

            # ── Exit logic (checked first) ──
            if in_trade:
                bars_held = i - entry_bar
                exit_reason = None

                # FIXED: 1. Mean reversion exit (z crosses 0) based on entry_z sign
                if entry_z >= Z_ENTRY and z <= Z_EXIT:
                    exit_reason = "mean_reversion"
                elif entry_z <= -Z_ENTRY and z >= -Z_EXIT:
                    exit_reason = "mean_reversion"

                # 2. Half-life timeout
                if exit_reason is None and bars_held >= hl_bars:
                    exit_reason = "halflife_timeout"

                # 3. Session-end forced exit at 15:28
                if exit_reason is None and bar_time >= FORCE_EXIT_TIME:
                    exit_reason = "session_end"

                if exit_reason:
                    exit_price = price
                    fees_info  = calc_zerodha_mis_fees(qty, entry_price, exit_price, is_long)
                    gross_pnl  = (exit_price - entry_price)*qty if is_long else (entry_price - exit_price)*qty
                    net_pnl    = gross_pnl - fees_info["total_fees"]
                    fees_mult  = (gross_pnl / fees_info["total_fees"]) if fees_info["total_fees"] > 0 else np.nan

                    trades.append(dict(
                        entry_bar=entry_bar, exit_bar=i,
                        duration_bars=bars_held,
                        entry_price=entry_price, exit_price=exit_price,
                        is_long=is_long, qty=qty,
                        gross_pnl=gross_pnl, net_pnl=net_pnl,
                        total_fees=fees_info["total_fees"],
                        fees_multiple=fees_mult,
                        exit_reason=exit_reason,
                        entry_z=entry_z
                    ))
                    in_trade = False
                    continue

            # ── Entry logic ──
            if not in_trade and bar_time < FORCE_EXIT_TIME:
                # FIXED: Mirrored entry directional logic depending on A/B lagger
                if z >= Z_ENTRY:        # spread too wide
                    if lagger_side == "a":
                        this_is_long = False  # A is overpriced, expect it to fall -> short A
                    else:
                        this_is_long = True   # B is underpriced relative to A, expect it to rise -> long B
                elif z <= -Z_ENTRY:     # spread too narrow
                    if lagger_side == "a":
                        this_is_long = True   # A is underpriced, expect it to rise -> long A
                    else:
                        this_is_long = False  # B is overpriced relative to A, expect it to fall -> short B
                else:
                    continue

                # Position sizing (based on raw price, Rs 10,000 isolated capital)
                this_qty = int(CAPITAL // price)
                if this_qty == 0:
                    continue  # stock too expensive (price > Rs 10,000)

                in_trade    = True
                entry_bar   = i
                entry_price = price
                entry_z     = z
                is_long     = this_is_long
                qty         = this_qty

        # If still in trade at end of data, force-close at last price
        if in_trade:
            exit_price = lag_prices[-1]
            fees_info  = calc_zerodha_mis_fees(qty, entry_price, exit_price, is_long)
            gross_pnl  = (exit_price - entry_price)*qty if is_long else (entry_price - exit_price)*qty
            net_pnl    = gross_pnl - fees_info["total_fees"]
            fees_mult  = (gross_pnl / fees_info["total_fees"]) if fees_info["total_fees"] > 0 else np.nan
            trades.append(dict(
                entry_bar=entry_bar, exit_bar=T-1, duration_bars=T-1-entry_bar,
                entry_price=entry_price, exit_price=exit_price,
                is_long=is_long, qty=qty,
                gross_pnl=gross_pnl, net_pnl=net_pnl,
                total_fees=fees_info["total_fees"], fees_multiple=fees_mult,
                exit_reason="data_end", entry_z=entry_z
            ))

        # ── Compute metrics ──
        n_trades = len(trades)
        if n_trades == 0:
            metrics = dict(
                total_trades=0, win_rate_pct=np.nan,
                total_gross_pnl=0.0, total_net_pnl=0.0, total_fees_paid=0.0,
                avg_fees_multiple=np.nan, avg_gross_pnl_per_trade=np.nan,
                avg_net_pnl_per_trade=np.nan, avg_trade_duration_minutes=np.nan,
                max_drawdown_pct=np.nan, sharpe_ratio=np.nan, calmar_ratio=np.nan,
                n_exits_mean_reversion=0, n_exits_halflife_timeout=0,
                n_exits_session_end=0, n_exits_data_end=0,
            )
        else:
            net_pnls   = [t["net_pnl"]   for t in trades]
            gross_pnls = [t["gross_pnl"] for t in trades]
            all_fees   = [t["total_fees"] for t in trades]
            durations  = [t["duration_bars"] for t in trades]
            fee_mults  = [t["fees_multiple"] for t in trades if not np.isnan(t["fees_multiple"])]

            cum_pnl = np.cumsum(net_pnls)
            peak    = np.maximum.accumulate(cum_pnl)
            dd      = peak - cum_pnl
            max_dd  = float(np.max(dd)) if len(dd) > 0 else 0.0
            max_dd_pct = (max_dd / CAPITAL) * 100.0

            # Daily P&L (group by date of exit bar)
            exit_dates = [timestamps[t["exit_bar"]].date() for t in trades]
            daily_pnl  = {}
            for d, p in zip(exit_dates, net_pnls):
                daily_pnl[d] = daily_pnl.get(d, 0) + p
            dpnl_arr = np.array(list(daily_pnl.values()))

            n_days = (timestamps[-1].date() - timestamps[WARMUP_BARS].date()).days + 1
            ann_return = (sum(net_pnls) / CAPITAL) * (250 / max(n_days, 1)) * 100

            sharpe = (np.mean(dpnl_arr) / np.std(dpnl_arr) * np.sqrt(250)
                      if len(dpnl_arr) > 1 and np.std(dpnl_arr) > 0 else np.nan)
            calmar = (ann_return / max_dd_pct
                      if max_dd_pct > 0 else np.nan)

            metrics = dict(
                total_trades=n_trades,
                win_rate_pct=100.0 * sum(1 for p in net_pnls if p > 0) / n_trades,
                total_gross_pnl=sum(gross_pnls),
                total_net_pnl=sum(net_pnls),
                total_fees_paid=sum(all_fees),
                avg_fees_multiple=np.mean(fee_mults) if fee_mults else np.nan,
                avg_gross_pnl_per_trade=np.mean(gross_pnls),
                avg_net_pnl_per_trade=np.mean(net_pnls),
                avg_trade_duration_minutes=float(np.mean(durations)),
                max_drawdown_pct=max_dd_pct,
                sharpe_ratio=sharpe,
                calmar_ratio=calmar,
                n_exits_mean_reversion=sum(1 for t in trades if t["exit_reason"]=="mean_reversion"),
                n_exits_halflife_timeout=sum(1 for t in trades if t["exit_reason"]=="halflife_timeout"),
                n_exits_session_end=sum(1 for t in trades if t["exit_reason"]=="session_end"),
                n_exits_data_end=sum(1 for t in trades if t["exit_reason"]=="data_end"),
            )

        runtime = time.time() - t_start
        result = {**base, **metrics,
                  "lagging_asset": lagging_asset,
                  "leader_asset" : leader_asset,
                  "total_bars"   : T,
                  "live_bars"    : T - WARMUP_BARS,
                  "data_start"   : str(timestamps[0]),
                  "data_end"     : str(timestamps[-1]),
                  "runtime_s"    : round(runtime, 3)}
        return result

    except Exception as ex:
        base["error"] = traceback.format_exc()[-500:]
        return base

print("Backtest engine defined.")


## Cell 8 — Run All Pairs


In [ ]:
results = []
total = len(s3_df)

for idx, row in s3_df.iterrows():
    print(f"[{idx+1:>3}/{total}] {row.symbol_a} - {row.symbol_b}  (HL={row.half_life_minutes:.1f}min)", end=" ... ", flush=True)
    res = backtest_pair(row, DB_PATH)
    results.append(res)
    if res.get("error"):
        print(f"ERROR: {res['error'][:80]}")
    elif res.get("skipped"):
        print(f"SKIPPED: {res['skip_reason']}")
    else:
        print(f"trades={res.get('total_trades',0)}  net_pnl=Rs{res.get('total_net_pnl',0):.1f}  calmar={res.get('calmar_ratio',float('nan')):.2f}")

print(f"\nAll {total} pairs processed.")


## Cell 9 — Merge Results + Build Ranked Output CSV


In [ ]:
results_df = pd.DataFrame(results)

# Merge Stage 2 cols we want in the final output
s2_cols = ["symbol_a","symbol_b","pearson_rho","stage1_rank",
           "Q_beta","Q_alpha","R","SNR_beta_R",
           "half_life_minutes","half_life_hours",
           "adf_pvalue","hurst_exponent","beta_mean","beta_std"]
s2_slim = s2_df[[c for c in s2_cols if c in s2_df.columns]]

final_df = results_df.merge(s2_slim, on=["symbol_a","symbol_b"], how="left",
                            suffixes=("","_s2"))

# Sort by Calmar Ratio descending (NaN last)
final_df = final_df.sort_values("calmar_ratio", ascending=False,
                                na_position="last").reset_index(drop=True)
final_df.insert(0, "stage3_rank", final_df.index + 1)

out_path     = "/kaggle/working/pairs_stage3_backtest.csv"
skipped_path = "/kaggle/working/skipped_stage3.csv"

skipped_df = final_df[final_df["skipped"] == True]
main_df    = final_df[final_df["skipped"] != True]

main_df.to_csv(out_path, index=False)
skipped_df.to_csv(skipped_path, index=False)

print(f"\n=== STAGE 3 SUMMARY ===")
print(f"Total pairs   : {len(final_df)}")
print(f"Backtested    : {len(main_df)}")
print(f"Skipped       : {len(skipped_df)}")
print(f"Errors        : {final_df['error'].astype(bool).sum()}")
print(f"\nTop 15 pairs by Calmar Ratio:")
cols_show = ["stage3_rank","symbol_a","symbol_b","half_life_minutes",
             "total_trades","win_rate_pct","total_net_pnl",
             "max_drawdown_pct","sharpe_ratio","calmar_ratio",
             "avg_fees_multiple","lagging_asset"]
cols_show = [c for c in cols_show if c in main_df.columns]
print(main_df[cols_show].head(15).to_string(index=False))

print(f"\nOutput  : {out_path}  ({os.path.getsize(out_path)/1024:.1f} KB)")
print(f"Skipped : {skipped_path}")


## Cell 10 — Publish to Kaggle Dataset


In [ ]:
import json, shutil
from kaggle.api.kaggle_api_extended import KaggleApi

os.environ["KAGGLE_USERNAME"] = "utkarshpatelthefirst"
os.environ["KAGGLE_KEY"]      = "fbef16329099428205f671dd5de8337b"

api = KaggleApi()
api.authenticate()

export_dir = "/kaggle/working/dataset_export"
os.makedirs(export_dir, exist_ok=True)
shutil.copy(out_path,     f"{export_dir}/pairs_stage3_backtest.csv")
shutil.copy(skipped_path, f"{export_dir}/skipped_stage3.csv")

meta = {
    "title"    : "Pairs Stage3 Backtest",
    "id"       : "utkarshpatelthefirst/pairs-stage3-backtest",
    "licenses" : [{"name": "CC0-1.0"}]
}
with open(f"{export_dir}/dataset-metadata.json", "w") as f:
    json.dump(meta, f, indent=2)

try:
    api.dataset_create_version(export_dir, version_notes="Stage3 v1.0-FRESH",
                               dir_mode="zip", quiet=False)
    print("Updated existing dataset.")
except Exception:
    api.dataset_create_new(export_dir, dir_mode="zip", quiet=False)
    print("Created new dataset.")

print("Published: https://www.kaggle.com/datasets/utkarshpatelthefirst/pairs-stage3-backtest")
